In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
#0-importation des paramètres configs.py et installation des librairies (à définir)
import config
import os

if os.path.exists("requirements.txt"):
    print("Installation des dépendances en cours...")
    !pip install -r requirements.txt --quiet
    print("Dépendances installées avec succès !")

Installation des dépendances en cours...
Dépendances installées avec succès !


In [10]:
#importation des donnees depuis S3 (si existent)
import src.importation_donnees_s3 

#importation des données DDSM depuis S3 vers le local
src.importation_donnees_s3.sync_s3_to_local(config.BUCKET_S3_DDSM, config.DDSM_PATH_LOCAL) 

#importation des données Vindr depuis S3 vers le local
os.makedirs(config.CHEMIN_DATA_VINDR, exist_ok=True)
src.importation_donnees_s3.sync_s3_to_local(config.BUCKET_S3_VINDR_NORMAL, config.VINDR_PATH_LOCAL_NORMAL) 
src.importation_donnees_s3.sync_s3_to_local(config.BUCKET_S3_VINDR_CANCER_BENIGN, config.VINDR_PATH_LOCAL_CANCER_BENIGN) 



Exécution de : mc mirror s3/lucasvital/stat_app/nouveau_dataset_mini_ddsm_700_700/ /home/onyxia/work/ot-domain-adaptation-mammography/projet/data/nouveau_dataset_mini_ddsm_700_700
Erreur lors de la synchronisation. Code : 2
Exécution de : mc mirror s3/lucasvital/stat_app/0-normal/ /home/onyxia/work/ot-domain-adaptation-mammography/projet/data/dataset_vindr/0-normal
┌───────┬─────────────┬──────────┬───────┐
│ Total │ Transferred │ Duration │ Speed │
│ 0 B   │ 0 B         │ 00m00s   │ 0 B/s │
└───────┴─────────────┴──────────┴───────┘
Synchronisation réussie !
Exécution de : mc mirror s3/lucasvital/stat_app/1-cancer_benign/ /home/onyxia/work/ot-domain-adaptation-mammography/projet/data/dataset_vindr/1-cancer_benign
┌───────┬─────────────┬──────────┬───────┐
│ Total │ Transferred │ Duration │ Speed │
│ 0 B   │ 0 B         │ 00m00s   │ 0 B/s │
└───────┴─────────────┴──────────┴───────┘
Synchronisation réussie !


In [ ]:
#1-importation_des_donnees_et_structuration (si donnees deja importes, passer cette etape)

In [6]:
#2-entrainement_miniddsm
from src.entrainement_ddsm import entrainement_model
from src.entrainement_ddsm import generation_donnees
import torch
from torchvision.models import ResNet18_Weights
import torchvision.models as models
import torch.nn as nn
import gc

# Generation des données
train_loader_DDSM, val_loader_DDSM, test_loader_DDSM = generation_donnees(
    chemin_data=config.DDSM_PATH_LOCAL,  
    limite_basse=config.LIMITE_BASSE,
    bs=config.BATCH_SIZE,
    resolution=config.RESOLUTION,
    choix_normalisation_image=config.CHOIX_NORMALISATION_IMAGE,
    brightness=config.BRIGHTNESS,
    contrast=config.CONTRAST,
    degrees=config.DEGREES,
    test_size=config.TEST_SIZE,
    val_size=config.VAL_SIZE,
    train_size=config.TRAIN_SIZE,
    numero_random=config.NUMERO_RANDOM
)

# Vérification GPU
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
else:
    print("Pas de GPU, utilisation du CPU")

# Création du modèle et configuration GPU
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Sequential(
    nn.Dropout(p=config.DROPOUT),  # Désactive 50% des neurones aléatoirement à chaque itération
    nn.Linear(model.fc.in_features, 2))

# Configuration GPU
device = torch.device("cuda:0")
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)  # ajout d'un weightdecay pour pénaliser l'overfitting
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=config.FACTOR, patience=config.PATIENCE, threshold=config.THRESHOLD)

#entraînement du modèle
entrainement_model(
    model=model,
    train_loader=train_loader_DDSM,  
    val_loader=val_loader_DDSM,      
    optimizer=optimizer,
    criterion=config.LOSS_FUNCTION,
    scheduler=scheduler,
    num_epochs=config.NUM_EPOCHS,
    patience=config.PATIENCE,
    patience_limite=config.PATIENCE_LIMIT,
    device=device,
    nom_modele_sauvegarde=config.NOM_MODELE_SAUVEGARDE,
    chemin_modele_local=config.CHEMIN_MODELE_LOCAL,
    chemin_modele_s3=config.CHEMIN_MODELE_S3
)

#nettoyage de la VRAM
gc.collect()
torch.cuda.empty_cache()

Nombre total d'images : 2922
Classes détectées : ['0-normal', '1-cancer_benign']
Distribution des classes : Counter({np.int64(0): 1873, np.int64(1): 1049})
accuracy random val : 0.6409993155373033
Train set: 2044
Validation set: 439
Test set: 439
Distribution dans train : Counter({np.int64(0): 1311, np.int64(1): 733})
Poids par classe : {np.int64(0): 1.559115179252479, np.int64(1): 2.7885402455661663}
GPU disponible : False
Pas de GPU, utilisation du CPU


RuntimeError: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.

In [ ]:
#3-evaluation_modele_sur_mini_ddsm

In [ ]:
#4-evaluation_modele_sur_vindr

In [ ]:
#5-evaluation_modele_sur_vindr_avec_transport_de_domaine (à la main et avec le modèle)

In [ ]:
#6-reentrainement_modele_avec_EWC

In [ ]:
#7-evluation_modele_EWC